In [4]:
import os
import numpy as np
from scipy.io import wavfile
from scipy.signal import butter, filtfilt
import matplotlib.pyplot as plt

# ---------- CONFIG ----------
LOWCUT, HIGHCUT = 20, 500   # Hz
PSW_THRESH_UV = 200          # µV
FIB_THRESH_UV = 10           # µV
ADC_SCALE = 32768            # 16-bit full scale
FULL_SCALE_UV = 800          # µV (±10 mV assumed)

def plot_single(file_path, start_ms=None, end_ms=None):
    try:
        # ---------- LOAD ----------
        fs, data = wavfile.read(file_path)
        if data.ndim > 1:
            data = np.mean(data, axis=1)

        # Convert to µV
        signal_uV = (data / ADC_SCALE) * FULL_SCALE_UV

        # ---------- FILTER ----------
        def butter_bandpass(lowcut, highcut, fs, order=4):
            nyq = 0.5 * fs
            low, high = lowcut / nyq, highcut / nyq
            b, a = butter(order, [low, high], btype='band')
            return b, a

        b, a = butter_bandpass(LOWCUT, HIGHCUT, fs)
        filtered = filtfilt(b, a, signal_uV)

        # ---------- DETECT THRESHOLD CROSSES ----------
        def detect_regions(signal, threshold):
            above = np.abs(signal) >= threshold
            regions = []
            in_region, start = False, None
            for i, val in enumerate(above):
                if val and not in_region:
                    in_region, start = True, i
                elif not val and in_region:
                    in_region = False
                    regions.append((start, i))
            if in_region:
                regions.append((start, len(signal)))
            return regions

        regions = detect_regions(filtered, PSW_THRESH_UV)

        # ---------- TIME AXIS ----------
        t = np.arange(len(filtered)) / fs * 1000  # ms

        # ---------- PLOT ----------
        fig, ax = plt.subplots(figsize=(20, 6))
        ax.plot(t, filtered, color='black', linewidth=1, label='Filtered EMG (µV)')

        # Determine time range
        x_min = start_ms if start_ms is not None else 0
        x_max = end_ms if end_ms is not None else t[-1]
        ax.set_xlim([x_min, x_max])

        # Horizontal PSW / FIB lines
        for y in [PSW_THRESH_UV, -PSW_THRESH_UV, FIB_THRESH_UV, -FIB_THRESH_UV]:
            color = 'red' if abs(y) == PSW_THRESH_UV else 'blue'
            ax.axhline(y=y, color=color, linestyle='--', linewidth=1)
            ax.text(x_max, y, f"{y:+.0f} µV", color=color,
                    verticalalignment='bottom' if y > 0 else 'top',
                    horizontalalignment='right')

        # Highlight threshold regions
        for start, end in regions:
            ax.axvspan(t[start], t[end], color='red', alpha=0.15)

        # Get y-axis limits for placing text labels
        y_min, y_max = ax.get_ylim()
        y_mid = (y_min + y_max) / 2

        # Grid: major every 1 s (1000 ms) - starting from x_min
        major_start = int(np.ceil(x_min / 1000) * 1000)
        major_ticks = np.arange(major_start, x_max + 1, 1000)
        ax.set_xticks(major_ticks)
        
        # Grid: minor every 10 ms (darker) - starting from x_min
        minor_start = int(np.ceil(x_min / 10) * 10)
        minor_ticks = np.arange(minor_start, x_max + 1, 10)
        ax.set_xticks(minor_ticks, minor=True)
        
        ax.grid(which='major', color='black', linestyle='-', linewidth=1, alpha=0.7)
        ax.grid(which='minor', color='grey', linestyle='-', linewidth=0.6, alpha=0.5)  # darker and solid

        # Add text labels for 10ms grid lines
        for x_pos in minor_ticks:
            if x_min < x_pos <= x_max:  # only show labels within visible range
                ax.text(x_pos, y_mid, f'{int(x_pos)}', 
                       rotation=90, verticalalignment='center', 
                       horizontalalignment='right',
                       fontsize=7, color='grey', alpha=0.7)

        ax.set_xlabel("Time (ms)")
        ax.set_ylabel("Amplitude (µV)")
        ax.set_title("Filtered EMG Signal (Absolute µV)")
        plt.tight_layout()

        # ---------- SAVE ----------
        deep_view_folder = os.path.join("Deep-View")
        os.makedirs(deep_view_folder, exist_ok=True)
        filename = os.path.basename(file_path).replace(".wav", ".png")
        plt.savefig(os.path.join(deep_view_folder, filename), dpi=300)
        plt.close(fig)
        print(f"Saved plot to: {os.path.join(deep_view_folder, filename)}")

    except Exception as e:
        print(f"Error processing {file_path}: {e}")


# ---------- USAGE EXAMPLES ----------
file_path = r"./labels-features/Spontanaktivität/filtered_HC280963.wav"

# Example 1: View 2-4 seconds (2000-4000 ms)
plot_single(file_path, start_ms=6000, end_ms=7000)

# Example 2: View first 2 seconds (0-2000 ms)
# plot_single(file_path, start_ms=0, end_ms=2000)

# Example 3: View entire signal (default)
# plot_single(file_path)

Saved plot to: Deep-View\filtered_HC280963.png
